In [1]:
# 🛠️ TOOL 1: Calculator
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression safely."""
    try:
        # Strip spaces to clean up basic math inputs
        clean_expression = expression.strip()
        return str(eval(clean_expression))
    except Exception:
        return "Error in calculation"

# 🛠️ TOOL 2: Keyword Extractor
def extract_keywords(text: str) -> list:
    """Extract unique keywords from text longer than 4 characters."""
    try:
        words = text.split()
        # Keep unique words longer than 4 characters, limited to top 5
        keywords = list(set([w.lower().strip(",.!?\"'") for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

In [ ]:
# 🤖 AGENT FUNCTION 
def agent(query: str) -> dict:
    """
    Single-Agent router that parses user intents, dispatches tasks 
    to dedicated tools, and returns clean, structured JSON payloads.
    """
    # Defensive programming: ensure we have a valid, clean string
    if not query or not isinstance(query, str):
        return {
            "type": "error",
            "result": "Invalid input query provided."
        }
        
    query_lower = query.lower()

    try:
        # 1. Routing Logic: Mathematical Calculations
        if "calculate" in query_lower:
            # Humanized touch: Strip out the activation word "calculate" to pass clean math to eval
            math_content = query.replace("Calculate", "").replace("calculate", "").strip()
            
            if not math_content:
                return {"type": "error", "result": "No mathematical expression provided."}
                
            output = calculator(math_content)
            
            if "Error" in output:
                return {"type": "error", "result": output}
            return {"type": "calculation", "result": int(output) if output.isdigit() else float(output)}

        # 2. Routing Logic: Keyword Extraction
        elif "keywords" in query_lower:
            # Humanized touch: Clean up common prompt wrappers to isolate the raw text target
            text_content = query
            prefixes_to_remove = ["extract keywords from", "extract keywords", "keywords"]
            
            for prefix in prefixes_to_remove:
                if text_content.lower().startswith(prefix):
                    text_content = text_content[len(prefix):].strip()
                    break
            
            if not text_content:
                return {"type": "error", "result": "No target text provided for extraction."}
                
            extracted_list = extract_keywords(text_content)
            return {"type": "keywords", "result": extracted_list}

        # 3. Routing Logic: General Knowledge Fallback
        else:
            # A simple, natural conversational fallback handler
            fallback_responses = {
                "what is machine learning?": "Machine learning is a subset of AI that allows systems to learn and improve from experience without being explicitly programmed.",
                "hi": "Hello! I am your smart assistant. How can I help you today with math or keyword extraction?",
                "hello": "Hello! I am your smart assistant. How can I help you today with math or keyword extraction?"
            }
            
            # Clean matching for known general questions
            clean_general = query.strip().lower()
            response = fallback_responses.get(
                clean_general, 
                f"I processed your query as a general request. You asked: '{query}'"
            )
            
            return {"type": "general", "result": response}

    except Exception as e:
        # Robust top-level error handling loop
        return {
            "type": "error",
            "result": f"An unexpected pipeline error occurred: {str(e)}"
        }

In [3]:
# 🧪 Test Cases
queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

print("=== RUNNING AUTOMATED PIPELINE TESTS ===\n")
for q in queries:
    print(f"Query: {q}")
    print(f"Response: {agent(q)}")
    print("-" * 50)

=== RUNNING AUTOMATED PIPELINE TESTS ===

Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': 25}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['transforming', 'artificial', 'industries', 'intelligence']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': 'Machine learning is a subset of AI that allows systems to learn and improve from experience without being explicitly programmed.'}
--------------------------------------------------


In [6]:
# 🎯 Interactive Mode
print("=== INTERACTIVE AGENT TERMINAL ACTIVE ===")
while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower().strip() == "exit":
        print("Shutting down agent pipeline. Goodbye!")
        break
    print("Response:", agent(user_input))
    print("-" * 50)

=== INTERACTIVE AGENT TERMINAL ACTIVE ===
Response: {'type': 'general', 'result': "I processed your query as a general request. You asked: 'what is machine learning ?'"}
--------------------------------------------------
Response: {'type': 'calculation', 'result': 4}
--------------------------------------------------
Response: {'type': 'general', 'result': "I processed your query as a general request. You asked: 'what is deep learning ?'"}
--------------------------------------------------
Shutting down agent pipeline. Goodbye!
